# Ch.1 座学：データ読み込み・集計（pandas）
### 講義時間：20分

---

> **講師メモ**：この座学のゴールは「データ分析の基本操作を実演で見せること」。  
> コードの細かい文法より「こういうことができる」という**体験**を優先する。  
> 各デモセルを実行しながら解説する。

---
## 【スライド1】この章で「できるようになること」

### データ分析の第一歩

```
実際の業務でも、分析は必ずここから始まります
```

| ステップ | 操作 | 確認すること |
|---------|------|-------------|
| 1 | データを読み込む | 何件ある？何列ある？ |
| 2 | データの形を把握する | どんな型？どんな値の範囲？ |
| 3 | 欠損値を確認・処理する | 空白のデータはどこにある？ |
| 4 | 集計・フィルタリングする | グループごとの傾向は？ |

---
## 【スライド2】pandas とは

### Pythonでデータ分析をするための最重要ライブラリ

- **DataFrame**（表形式データ）を扱うための道具箱
- Excelのような「行×列」のデータをPythonで操作できる
- データサイエンティストが**毎日使う**基本ツール

```
Excel でできること → pandas でもできる
Excel でできないこと → pandas ならできる（大量データ・自動化・再現性）
```

In [ ]:
# ライブラリの読み込み
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine

# データ準備
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target
df['品種'] = df['target'].map({0: 'Barolo', 1: 'Grignolino', 2: 'Barbera'})

print('データ準備完了！')
print(f'→ {len(df)} 件のワインデータ、{len(df.columns)} 列')

---
## 【スライド3】データの基本確認：3つのメソッド

### データを手に入れたら、まず必ずこの3つを実行する

| メソッド | 何がわかるか |
|---------|-------------|
| `df.head()` | 先頭数行を見て「どんなデータか」を把握 |
| `df.info()` | 列名・データ型・欠損値の有無を確認 |
| `df.describe()` | 平均・最大・最小・標準偏差などの統計量 |

In [ ]:
# デモ①：head() で先頭5行を確認
print('=== df.head() ===')
print('→ データの「顔」を見る。列名・値の形式を把握する。')
df.head()

In [ ]:
# デモ②：info() で型・欠損値を確認
print('=== df.info() ===')
print('→ 列ごとのデータ型と Non-Null Count（欠損がない行数）を確認。')
print()
df.info()

In [ ]:
# デモ③：describe() で統計量を確認
print('=== df.describe() ===')
print('→ count（件数）・mean（平均）・std（標準偏差）・min・max が一覧で見える。')
print()
df.describe().round(2)

---
## 【スライド4】欠損値とは何か

### 現実のデータには必ず「空白」がある

```
欠損値（NaN = Not a Number）
  → データが記録されていない、入力ミス、センサー故障 などが原因
  → そのままモデルに入れるとエラーになる
```

### 欠損値の処理方針

| 方法 | 使う場面 | リスク |
|------|---------|--------|
| **削除**（`dropna`） | 欠損が少量かつランダムな場合 | データ量が減る |
| **平均値で補完**（`fillna`） | 数値データで欠損が一定量ある場合 | 分布が変わる可能性 |
| **グループ別平均で補完** | グループ間に差がある場合 | 少し複雑 |

> **「どれが正解か」は一概には言えない。データの性質とビジネス要件次第。**

In [ ]:
# デモ：欠損値を人工的に作って確認
df_demo = df.copy()
np.random.seed(42)
for col in ['alcohol', 'color_intensity']:
    idx = np.random.choice(df_demo.index, size=15, replace=False)
    df_demo.loc[idx, col] = np.nan

print('=== 欠損値の確認 ===')
missing = df_demo.isnull().sum()
print(missing[missing > 0])
print()
print('=== 欠損率（%） ===')
rate = (df_demo.isnull().sum() / len(df_demo) * 100).round(1)
print(rate[rate > 0])

In [ ]:
# デモ：平均値で補完
df_filled = df_demo.copy()
for col in ['alcohol', 'color_intensity']:
    mean_val = df_filled[col].mean()
    df_filled[col] = df_filled[col].fillna(mean_val)
    print(f'{col}: 平均 {mean_val:.3f} で補完')

print(f'\n補完前の欠損件数: {df_demo.isnull().sum().sum()}')
print(f'補完後の欠損件数: {df_filled.isnull().sum().sum()}')

---
## 【スライド5】フィルタリングと集計

### データを「絞り込む」「まとめる」の2操作が基本

```python
# フィルタリング：条件に合う行を抽出
df[df['列名'] > 値]

# 集計：グループごとに統計量を計算
df.groupby('グループ列')['集計列'].mean()
```

> **実務イメージ**：「関東だけのデータで月次売上を比較する」→ フィルタ後にgroupby

In [ ]:
# デモ：フィルタリング
print('=== アルコール度数 13.5 以上のワイン ===')
high_alc = df[df['alcohol'] >= 13.5]
print(f'全体 {len(df)} 件 → 条件に合う {len(high_alc)} 件')
print(high_alc[['品種', 'alcohol', 'color_intensity']].head(5))

In [ ]:
# デモ：groupby 集計
print('=== 品種ごとの主要特徴量の平均 ===')
group = df.groupby('品種')[['alcohol', 'color_intensity', 'proline']].mean().round(2)
print(group)
print()
print('→ Barolo はアルコール度数もプロリンも高い傾向が見える')

---
## 【スライド6】まとめ

### Ch.1 で覚えておくこと

| 操作 | コード | ポイント |
|------|--------|----------|
| 形を確認 | `head()` / `info()` / `describe()` | この3つを最初に必ず実行 |
| 欠損値確認 | `isnull().sum()` | 欠損があるか確認してから進む |
| 欠損値補完 | `fillna(平均値)` | 削除か補完かはデータ次第 |
| フィルタ | `df[df['列'] > 値]` | 条件に合う行だけ取り出す |
| 集計 | `groupby('列').mean()` | グループごとの傾向を掴む |

### 次は「可視化」へ
> 集計で見えてきた数値を、グラフにして「直感的に」理解できるようにします